In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/scaling-admission-cost/agentic-scaling-lab/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 05 · Hosted API or your own GPUs — the Mistral provider and a vLLM fleet

**What you will learn.** Notebooks 01–04 calculated the anchor scenario on a hosted API. On a hosted API, the only
way to pay for tokens is per token. An open-weight model adds a second way: you run it yourself on a vLLM fleet. You
size the fleet for the peak, and you pay for it by the hour, when it is busy and when it is not.

This notebook sends the same 100,000 conversations a day to Mistral's API and to a fleet of Ministral 14B replicas.
It shows these things:

- what one replica delivers,
- how many replicas you need,
- the GPU price at which the fleet beats the API,
- what an overload looks like on a fleet: no 429s, only everyone slower.

The code is in `scalelab/mistral.py` (the demand and the money), `scalelab/serving.py` (one replica), the
`ServerPool` and `HybridBackend` backends in `scalelab/model.py`, and `scalelab/sim.py`. The primer is
`docs/mistral/01-scaling-primer.md`.

> **In a design review.** *"The unit of work is a turn, and 100k conversations a day is about 14 million tokens a
> minute at peak. On the API, that is a 60-RPS / 19 M-TPM limit that I must ask for. On our own GPUs, it is eleven
> H100s, sized for the peak. At this volume, the two cost about the same on committed GPUs. Thus the data-residency
> requirement decides, and the arithmetic gives the price of that choice."*

Each number here comes from a simulation or from an estimate from first principles, with a label that says so. The
prices are list prices dated 2026-09-19. Make sure that they are correct before you quote them.

In [ ]:
import os, sys, asyncio, random
sys.path[:0] = [".", ".."]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nbutil import todo, check, acheck, latency_cdfs, timeline
from scalelab.clock import CLOCK
%matplotlib inline
pd.set_option("display.width", 160)
CLOCK.reset(0.02)   # 50x faster than real time

## 1. The same demand on Mistral's API

The chain is the same as in notebook 01: from conversations to turns, to calls, to tokens per minute. Little's law
gives what is in flight. Only the price per token and the limit change. Mistral shows its paid-tier limits only in
the console. Thus the capacity plan **is** the rate-limit request that you send.

In [ ]:
from scalelab.mistral import Scenario, plan, cost_per_call, fleet, FREE_TIER
from scalelab.serving import replica, GPU_USD_PER_HOUR
s = Scenario()          # the anchor scenario: 100k conversations/day, Mistral Small 4 with 10 % on Medium 3.5
p = plan(s)
tokens = pd.DataFrame(p["tokens"]).T
print("peak total TPM (M):", round(tokens.total_tpm["peak"] / 1e6, 1), "| vs the assumed limit:", round(tokens.vs_limit["peak"], 2), "x")
print("the request to send:", p["hosted"]["limit_request"])
pd.DataFrame({"$/conversation": p["hosted"]["cost_per_conversation"], "$/month": p["hosted"]["monthly_usd"]}).round(4)

Mistral bills cached input at 10 %. The Priority Tier is +75 %, and a regional endpoint is +10 % (2026-09-19,
verify). Route and cache first, because every later percentage applies to the new baseline.

## 2. What one replica delivers

A vLLM replica has two budgets: memory and the time per decode step.

**Memory**: the weights plus the KV cache. Thus the KV bytes per token decide how many calls can be resident.
Grouped-query attention in Ministral 14B uses 160 KiB per token. Latent attention in Small 4 uses 22.5 KiB.

**Time per decode step**: each step moves the weights that it touches, plus the KV of each sequence that runs,
through HBM. Thus the time per output token (TPOT) increases with the batch. At the same time, the aggregate
tokens/s increases until the step is compute-bound.

*The latency that you want is the batch that you can afford.* These are first-principles estimates. Layer 04's
`minengine.perf` is the reference model of the step time. Replace these estimates with measurements from
`vllm bench serve` on the GPU that you get.

In [ ]:
pd.DataFrame([replica(m, g).describe() for m, g in [("ministral-14b", "h100"), ("ministral-14b", "l40s"),
                                                      ("mistral-small-4", "h100"), ("mistral-small-4", "h200"), ("mistral-large-3", "h200")]]).set_index("model")

In [ ]:
ctx = s.input_tokens + s.output_tokens
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for key, gpu in [("ministral-14b", "h100"), ("mistral-small-4", "h100")]:
    r = replica(key, gpu); batches = range(1, 129)
    axes[0].plot(batches, [r.tpot(b, ctx) * 1e3 for b in batches], label=f"{r.model.name} on {r.tp}×{gpu.upper()}")
    axes[1].plot(batches, [r.tokens_per_s(b, ctx) for b in batches], label=f"{r.model.name} on {r.tp}×{gpu.upper()}")
axes[0].axhline(s.target_tpot_s * 1e3, ls="--", c="grey"); axes[0].set_ylabel("TPOT (ms, estimated)"); axes[0].set_xlabel("sequences in the batch")
axes[1].set_ylabel("aggregate tokens / s (estimated)"); axes[1].set_xlabel("sequences in the batch")
for ax in axes: ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout()

## 3. The fleet, and where it beats the API

Calculate the size of the fleet for the peak at the target TPOT. Add headroom, and never use fewer than two
replicas. Calculate the price of the GPUs by the hour.

Both bills increase with volume. The bill of the API increases linearly, and the bill of the fleet increases in
steps. Thus the break-even is not a volume but a **GPU price**. It is the hourly rate at which the peak-sized fleet
costs the same as the bill of the API for the same conversations. Volume is important only at the bottom, where the
two-replica floor is not yet amortised.

In [ ]:
f = p["self_hosted"]
print(f"{f['model']} on {f['gpu']}: batch {f['batch']} → TPOT {f['tpot_ms']:.0f} ms, {f['tok_s_per_replica']:,.0f} tok/s per replica; "
      f"a call takes {f['call_seconds']:.1f} s, a turn {f['turn_seconds']:.1f} s (hosted: {s.turn_seconds:g} s)")
print("replicas (avg / peak / incident):", f["replicas"], "→ in-flight cap for the peak fleet:", round(f["max_inflight_turns_peak_fleet"]))
print(f"break-even GPU price vs the hosted planning mix: ${f['breakeven_gpu_usd_per_hour']:.2f} per GPU-hour")
pd.DataFrame({"$/GPU-h": GPU_USD_PER_HOUR[s.gpu], "peak fleet $/month": f["monthly_usd_peak_by_price"], "$/conversation": f["cost_per_conversation_by_price"],
              "vs hosted": f["vs_hosted_by_price"], "floor conversations/day": f["floor_conversations_per_day_by_price"]}).round(4)

In [ ]:
hosted_per_conv = p["hosted"]["cost_per_conversation"][p["hosted"]["planning_mix"]]
volumes = np.arange(5_000, 300_001, 5_000)
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(volumes, volumes * hosted_per_conv * 30.4 / 1e3, label="Mistral API (planning mix, cached)", lw=2)
for price in ("aws on-demand", "gcp 3-year", "neocloud"):
    fleets = [fleet(Scenario(conversations_per_day=int(v), gpu_price=price))["monthly_usd"]["peak"] / 1e3 for v in volumes]
    ax.step(volumes, fleets, where="post", label=f"fleet sized for peak, {price} (${GPU_USD_PER_HOUR[s.gpu][price]:.2f}/GPU-h)")
ax.set_xlabel("conversations per day"); ax.set_ylabel("k$ / month"); ax.set_title("The API is a line; a fleet is a staircase whose slope is the GPU price"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout()

## 4. On a fleet nobody is refused — everybody is slower

The next cell runs one Ministral 14B replica on an H100 (simulated). Send $k$ calls at the same time. You get no 429 and no
error. The batch increases, the time per token increases with it, and every call in the batch pays. After the batch
at which the target TPOT holds (dashed line), the latency per call increases faster than the throughput.

In [ ]:
from scalelab.model import ServerPool
rows = []
for k in (1, 8, 16, 24, 32, 48, 64, 96):
    CLOCK.reset(0.02)
    srv = ServerPool(replica("ministral-14b", "h100"), replicas=1, context_tokens=5_200, shared_prefix_tokens=3_000)
    rng = random.Random(0); t0 = CLOCK.now()
    lat = await asyncio.gather(*(srv.serve(5_000, 3_000, 300, rng) for _ in range(k)))
    rows.append({"concurrent calls": k, "p50 call latency (s)": np.median([l for l, _ in lat]), "tokens/s served": k * 300 / (CLOCK.now() - t0), "errors": srv.rejected})
df = pd.DataFrame(rows).set_index("concurrent calls").round(2); print(df)
ax = df[["p50 call latency (s)"]].plot(figsize=(6.5, 3.4), marker="o", title="One replica: latency vs concurrency (simulated)")
ax.axvline(srv.target_batch, ls="--", c="grey"); ax.set_ylabel("seconds"); ax.grid(alpha=.3); plt.tight_layout()

Thus, on a fleet, the admission controller reads one more signal: **saturation**. Saturation is running + waiting,
against the batch that the latency target permits. Level 1 is at 0.8, and level 2 at 1.0. The saturation detector
of the Kubernetes inference gateway uses the same idea: queue depth ≥ 5 or KV usage ≥ 0.8 (verify). On the API, that
signal is the use of the pool.

## 5. The load test, on each backend

The load test runs the same users against the API, the fleet and the hybrid. The hybrid uses the fleet first, and
the API when the fleet is saturated. `make_setup(mode)` selects the backend. If you call it without a mode, it reads
`SCALELAB_BACKEND` (`hosted`, `local` or `hybrid`) and `SCALELAB_PROVIDER` (`gemini` or `mistral`).

Thus `SCALELAB_BACKEND=local jupyter lab` runs the load test of notebook 04 against a fleet, with no edit to the
notebook. The exercise (d) of notebook 04 then compares the result with the target of the fleet, as exercise (f) of this notebook
does.

In [ ]:
from scalelab.sim import make_setup, simulate, compare
results = {}
for name, mode, kw, users in [("API, moderate (20 users)", "hosted", dict(provider="mistral", pool_tpm=3_000_000), 20),
                              ("API, overload, naive", "hosted", dict(provider="mistral", pool_tpm=3_000_000, naive=True), 120),
                              ("API, overload, degrade + cap", "hosted", dict(provider="mistral", pool_tpm=3_000_000), 120),
                              ("fleet, moderate (20 users)", "local", dict(replicas=2), 20),
                              ("fleet, overload, naive", "local", dict(replicas=2, naive=True), 120),
                              ("fleet, overload, degrade + cap", "local", dict(replicas=2), 120),
                              ("hybrid, overload", "hybrid", dict(replicas=2, pool_tpm=3_000_000), 120)]:
    CLOCK.reset(0.02)
    results[name] = await simulate(make_setup(mode, **kw), users=users, duration_s=90, think_s=5)
summary = compare(results).round(4)
summary[["turns", "completed", "shed_rate", "p50_s", "p95_s", "pushback_calls", "failed", "degraded_share", "max_batch", "api_share_of_calls", "cost_per_turn_usd"]]

In [ ]:
latency_cdfs({k: v for k, v in results.items() if not k.startswith("API")}, "Turn latency — the fleet and the hybrid (simulated)")

In [ ]:
timeline(results["fleet, overload, naive"], "fleet, naive: no errors, no 429s — the batch climbs and everyone slows down")

In [ ]:
timeline(results["hybrid, overload"], "hybrid: the batch holds near the target; the excess goes to the API")

On the API, the naive run is the feedback loop of notebook 04. 429s cause retries, the retries make the turns
longer, and the longer turns put more turns in flight. On the fleet, the loop takes its second form. It has zero
errors, zero pushback, a batch far past the target, and a p95 well above the moderate run. The dashboard is green
while every user waits.

The cap and the saturation signal put a limit on the batch. The hybrid keeps the batch at the target and pays the
API for the rest. `cost_per_turn_usd` shows the cost of that.

## 6. From measurements to vLLM and Kubernetes settings

Take the throughput per replica at the target batch, and the turn duration, from the moderate runs. Apply Little's
law at the planned peak. Add headroom. Then divide by the quantity that one pod or one replica can hold.

In [ ]:
fleet_turn = results["fleet, moderate (20 users)"].summary()["p95_s"]
print(f"measured p95 turn on the fleet: {fleet_turn:.1f} s (simulated; the plan assumed {f['turn_seconds']:.1f} s)")
for level in ("average", "peak", "incident"):
    turns_s, calls_s = p["rates"][level]["turns_per_s"], p["rates"][level]["calls_per_s"]
    pods = np.ceil(turns_s * fleet_turn * 1.4 / 80)
    by_tokens = calls_s * s.output_tokens / f["tok_s_per_replica"]          # decode throughput
    by_seqs = calls_s * f["call_seconds"] / f["batch"]                       # Little: calls in flight ÷ batch per replica
    print(f"{level:9s} {turns_s:5.1f} turns/s → {pods:2.0f} orchestrator pods (80 turns each, ×1.4); "
          f"{calls_s:5.1f} calls/s → {by_tokens:4.1f} replicas by tokens/s, {by_seqs:4.1f} by batch → {np.ceil(max(by_tokens, by_seqs) * 1.3):2.0f} vLLM replicas (×1.3)")

| Setting | Value (Ministral 14B on H100) | Why |
|---|---|---|
| vLLM replicas | 11 at peak (static), KEDA to 14 for headroom | Tokens/s per replica at the target batch. The cold start of a GPU replica takes minutes. Thus you supply the peak capacity in advance, and you do not autoscale it. |
| `--max-num-seqs` | 48 (2× the target batch of 24) | It gives room above the target for bursts. Above the target, you give up TPOT. |
| `--max-num-queued-reqs` | ≈ 12 per replica | It changes a queue into a 503 that the gateway can act on. Without it, vLLM keeps requests in its queue forever. |
| `--max-model-len` | 16,384 | It frees KV memory, compared with the long default of the model. The turn never needs more. |
| `--kv-cache-dtype fp8` | on | It divides the KV bytes per token by two. Thus it gives about twice the resident sequences. |
| in-flight cap (gateway) | about 290 turns for the peak fleet, 175 on a 20 M TPM API limit | Little's law from the capacity of the fleet or of the limit. Adjust it from the load test. |
| KEDA (fleet) | `vllm:num_requests_waiting` > 5 or `vllm:kv_cache_usage_perc` > 0.8 | The saturation thresholds of the inference gateway. |

The flags and thresholds are as of vLLM 0.29–0.30 and the Gateway API Inference Extension, September 2026 (verify).
The last section of `docs/02-reference-architecture.md` gives the Kubernetes reference architecture for these
settings.

## 7. What breaks first

In [ ]:
pd.DataFrame(p["breaks_first"]).round(2)

## Your turn

Fill in each function (replace the `todo()` call). Then run the check cell that comes after it. The practice
notebook runs end to end when the exercises have no change. In the solutions notebook, every check passes. The
checks compare your function with the model of the lab over many inputs, not one.

#### (a) KV cache and resident sequences

Calculate the bytes per token for grouped-query attention:

$$
2\ (\text{K and V}) \times \text{layers} \times \mathrm{kv\_heads} \times \mathrm{head\_dim} \times \text{bytes per element}.
$$

Then calculate the number of sequences that a KV budget holds. The budget holds the shared prefix one time. Each
sequence pays for its own tokens. The number is never more than `max_num_seqs`.

In [ ]:
def kv_bytes(layers, kv_heads, head_dim, bytes_per_elem=2):
    return todo()

def max_sequences(kv_budget_gb, kv_bytes_per_token, context_tokens, prefix_tokens=0, max_num_seqs=128):
    return todo()

In [ ]:
def _a():
    from scalelab.serving import kv_bytes_per_token
    for layers, heads, dim, b in [(40, 8, 128, 2), (34, 8, 128, 2), (61, 4, 192, 1), (12, 2, 64, 4)]:
        assert kv_bytes(layers, heads, dim, b) == kv_bytes_per_token(layers, heads, dim, b), (layers, heads, dim, b)
    for m, g in [("ministral-14b", "h100"), ("ministral-8b", "l40s"), ("mistral-small-4", "h200"), ("mistral-large-3", "b200")]:
        for fp8 in (False, True):
            r = replica(m, g, kv_dtype_bytes=1 if fp8 else 2)
            for ctx, prefix in [(5_200, 0), (5_200, 2_700), (32_000, 3_000), (1_000, 0)]:
                got = max_sequences(r.kv_budget_gb, r.model.kv_bytes_per_token * r.kv_dtype_bytes / 2, ctx, prefix)
                assert got == r.max_seqs(ctx, prefix), f"{m} on {g}, fp8 KV={fp8}, {ctx} tokens with a {prefix} prefix: {got} vs {r.max_seqs(ctx, prefix)}"
    r = replica("ministral-14b", "h100")
    print(f"   Ministral 14B on one H100: {r.max_seqs(5_200)} calls of 5.2k tokens resident without prefix caching, "
          f"{r.max_seqs(5_200, 2_700)} with (the max_num_seqs cap), {replica('ministral-14b', 'h100', kv_dtype_bytes=1).max_seqs(5_200)} with FP8 KV")
check("a: KV cache", _a)

#### (b) The break-even GPU price

The peak-sized fleet uses `gpus × 730` GPU-hours a month to serve `conversations_per_day × 30.4` conversations. At
what hourly GPU price is that equal to the price per conversation of the API? Also, below what daily volume is a
minimum fleet of `min_gpus` GPUs not amortised at a given price?

In [ ]:
def breakeven_gpu_usd_per_hour(gpus, conversations_per_day, hosted_usd_per_conversation):
    return todo()

def floor_conversations_per_day(min_gpus, usd_per_gpu_hour, hosted_usd_per_conversation):
    return todo()

In [ ]:
def _b():
    for kw in [{}, dict(conversations_per_day=30_000), dict(conversations_per_day=250_000, premium_share=0.3),
               dict(open_model="mistral-small-4", gpu="h200", gpu_price="neocloud"), dict(output_tokens=600, turns_per_conversation=9)]:
        sc = Scenario(**kw); pl = plan(sc); fl = pl["self_hosted"]
        mix = pl["hosted"]["cost_per_conversation"][pl["hosted"]["planning_mix"]]
        be = breakeven_gpu_usd_per_hour(fl["gpus"]["peak"], sc.conversations_per_day, mix)
        assert abs(be - fl["breakeven_gpu_usd_per_hour"]) < 1e-6 * be, (kw, be, fl["breakeven_gpu_usd_per_hour"])
        tp = fl["gpus"]["peak"] // fl["replicas"]["peak"]
        for price_key, usd in GPU_USD_PER_HOUR[sc.gpu].items():
            want = fl["floor_conversations_per_day_by_price"][price_key]
            assert abs(floor_conversations_per_day(sc.min_replicas * tp, usd, mix) - want) < 1e-6 * want, (kw, price_key)
    f, mix = p["self_hosted"], p["hosted"]["cost_per_conversation"][p["hosted"]["planning_mix"]]
    be = breakeven_gpu_usd_per_hour(f["gpus"]["peak"], s.conversations_per_day, mix)
    print(f"   the fleet beats the API below ${be:.2f}/GPU-hour — at AWS on-demand H100 prices ($6.88) it loses, at 3-year commitments "
          f"($4.86–5.40) it is level, at neocloud prices ($3.75) it wins; at $3.75 two replicas are amortised from "
          f"{floor_conversations_per_day(2, 3.75, mix):,.0f} conversations/day (list prices 2026-09-19, verify)")
check("b: break-even price", _b)

#### (c) Replicas needed

Calculate the replicas for two constraints, and take the larger result:

- decode throughput: calls/s × output tokens ÷ tokens/s per replica,
- resident calls: calls/s × call seconds ÷ batch per replica (Little's law).

Then multiply by the headroom and round up. Never return a value below `minimum`.

In [ ]:
def replicas_needed(calls_per_s, output_tokens, tok_s_per_replica, call_seconds, batch, headroom=1.3, minimum=2):
    return todo()

In [ ]:
def _c():
    for kw in [{}, dict(conversations_per_day=5_000), dict(conversations_per_day=400_000), dict(open_model="mistral-small-4", gpu="h200"),
               dict(target_tpot_s=0.035, headroom=1.5, min_replicas=3), dict(gpu="l40s", target_tpot_s=0.04, output_tokens=400)]:
        sc = Scenario(**kw); fl = fleet(sc); pl = plan(sc)
        for level in ("average", "peak", "incident"):
            got = replicas_needed(pl["rates"][level]["calls_per_s"], sc.output_tokens, fl["tok_s_per_replica"], fl["call_seconds"], fl["batch"],
                                  sc.headroom, sc.min_replicas)
            assert got == fl["replicas"][level], f"{kw or 'anchor'} at {level}: {got} replicas, the model sizes {fl['replicas'][level]}"
    print("   anchor scenario:", p["self_hosted"]["replicas"])
check("c: replicas", _c)

#### (d) The in-flight cap from the fleet

Calculate how many turns can be in flight before the fleet goes past its target batch:

$$
\frac{\text{replicas} \times \text{batch} \times \text{turn seconds}}{\text{calls per turn} \times \text{call seconds}}.
$$

In [ ]:
def cap_from_fleet(replicas, batch, turn_seconds, calls_per_turn, call_seconds):
    return todo()

In [ ]:
def _d():
    for kw in [{}, dict(conversations_per_day=40_000), dict(calls_per_turn=3.1, tool_seconds_per_turn=2.5), dict(gpu="h200", target_tpot_s=0.03)]:
        sc = Scenario(**kw); fl = fleet(sc)
        got = cap_from_fleet(fl["replicas"]["peak"], fl["batch"], fl["turn_seconds"], sc.calls_per_turn, fl["call_seconds"])
        assert abs(got - fl["max_inflight_turns_peak_fleet"]) < 1e-6 * got, (kw or "anchor", got, fl["max_inflight_turns_peak_fleet"])
    print(f"   the peak fleet holds about {p['self_hosted']['max_inflight_turns_peak_fleet']:.0f} turns in flight before it passes its target batch")
check("d: cap from the fleet", _d)

#### (e) The spill-over decision

Route a model call with these rules:

- Return `"fleet"` while the saturation of the fleet is below `spill_at`.
- Otherwise, return `"api"` if the call carries no personal data (the residency policy) and the breaker of the API is closed.
- Otherwise, return `"degrade"` (stay on the fleet with shorter answers).

Priority callers (≤ 2) always get the fleet.

In [ ]:
def route(saturation, has_pii, api_breaker_closed, priority=5, spill_at=0.9):
    return todo()

In [ ]:
def _e():
    assert route(0.5, True, True) == "fleet"
    assert route(0.95, False, True) == "api"
    assert route(0.95, True, True) == "degrade"           # personal data never leaves your environment
    assert route(0.95, False, False) == "degrade"         # the API is broken: stay home
    assert route(1.5, True, True, priority=1) == "fleet"  # priority traffic gets the GPUs
    assert route(0.7, False, True, spill_at=0.6) == "api" and route(0.9, False, True) == "api"   # the threshold is inclusive
    for sat in np.linspace(0, 2, 41):                     # below the threshold nothing else matters
        if sat < 0.9:
            assert {route(sat, pii, ok) for pii in (True, False) for ok in (True, False)} == {"fleet"}, sat
check("e: spill-over", _e)

#### (f) Choose the cap

Select an in-flight cap for two replicas with 120 users. The cap must give p95 ≤ 7.5 s (simulated) and no failed
turn. Keep the shed rate under 45 %. The check runs a simulation with your value.

In [ ]:
MY_CAP = None   # an integer

In [ ]:
async def _f():
    if MY_CAP is None:
        todo()
    CLOCK.reset(0.02)
    r = await simulate(make_setup("local", replicas=2, max_inflight=MY_CAP), users=120, duration_s=60, think_s=5)
    s_ = r.summary()
    print(f"   cap {MY_CAP}: p95 {s_['p95_s']:.1f} s, shed {s_['shed_rate']:.0%}, failed {s_['failed']}, max batch {s_['max_batch']}")
    assert s_["p95_s"] <= 7.5 and s_["failed"] == 0 and s_["shed_rate"] < 0.45
await acheck("f: choose the cap", _f)

## Takeaways for the design review

- Hosted or self-hosted is a price question with a residency answer. Quote the break-even GPU price (≈ $5 per
  GPU-hour here: on-demand loses, committed is level, neoclouds win). Then say which requirement decides.
- One replica is a KV budget and a bandwidth budget. The batch that you run *is* your latency. Calculate the size of
  the fleet for the peak at the target TPOT. Then calculate its price by the hour.
- The naive fleet does not fail. It becomes slow for everyone, and the dashboard stays green. Put a cap on the queue.
  Monitor the batch. Send saturation to admission control. Let the API take the spill-over where the data policy
  permits it.

## Verify before the design review

Examine these facts:

- The Mistral list prices and model ids (2026-09-19 in `scalelab/mistral.py`).
- The paid-tier limits in the console.
- The replica estimates, against `vllm bench serve` on the GPUs that you get.
- The hourly GPU prices in your region.
- The vLLM flags and defaults (`max_num_seqs` 128, `--max-num-queued-reqs`).
- The saturation thresholds of the inference gateway.